In [ ]:
!pip install anthropic textstat pandas scikit-learn

import os
import pandas as pd
import textstat
import anthropic
from anthropic import HUMAN_PROMPT, AI_PROMPT

HUMAN_PROMPT = anthropic.HUMAN_PROMPT
AI_PROMPT    = anthropic.AI_PROMPT

client = anthropic.Client(api_key="")

In [3]:
base = '/Users/led7283981/Downloads/mimic-iii-clinical-database-1.4'
os.chdir(base)

admissions = pd.read_csv('ADMISSIONS.csv')
patients   = pd.read_csv('PATIENTS.csv')
diagnoses  = pd.read_csv('DIAGNOSES_ICD.csv')
noteevents = pd.read_csv('NOTEEVENTS.csv')

/var/folders/1h/jdd4m_cx7rd577m1v41q0nsh0000gn/T/ipykernel_25380/1639539200.py:7: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  noteevents = pd.read_csv('NOTEEVENTS.csv')


In [4]:
try:
    response = client.messages.create(
        model="claude-3-5-sonnet-20240620",
        max_tokens=10,
        messages=[{"role": "user", "content": "Ping, Claude!"}]
    )
    print("Claude connectivity OK:", response.content[0].text)
except Exception as e:
    print("Claude connectivity error:", e)

Claude connectivity OK: Hello! I'm here and ready to help.


In [5]:
file_path = '/Users/led7283981/Downloads/mimic-iii-clinical-database-1.4/DIAGNOSES_ICD.csv'
df = pd.read_csv(file_path)

In [17]:
code_col = (
    'ICD9_CODE' if 'ICD9_CODE' in df.columns
    else next((col for col in df.columns
               if 'icd' in col.lower() and 'code' in col.lower()), None)
)

if code_col is None:
    raise KeyError(
        f"No ICD code column (e.g., 'ICD9_CODE') found in {file_path}. "
        f"Available columns: {df.columns.tolist()}"
    )

In [25]:
first_code = str(df.at[0, code_col]).lstrip('0')
print(f"First ICD-9 code in the dataset ({code_col}): {first_code}")

First ICD-9 code in the dataset (ICD9_CODE): 1166


In [26]:
resp = client.messages.create(
    model="claude-3-5-sonnet-20240620",
    max_tokens=20,
    messages=[
        {"role": "user", "content": HUMAN_PROMPT + f"What does ICD-9 code {first_code} generally represent in medical terms?" + AI_PROMPT}
    ]
)

In [27]:
print("Claude Response:", resp.content[0].text)

Claude Response: I apologize, but there appears to be an error in your question. ICD-9 


In [36]:
for i in range(3):
    note_text = noteevents['TEXT'].dropna().iloc[i]
    print(f"\n--- Sample Note {i+1} ---\n")
    print(note_text[:1000]) 
    print("\n[...truncated]\n")


--- Sample Note 1 ---

Admission Date:  [**2151-7-16**]       Discharge Date:  [**2151-8-4**]


Service:
ADDENDUM:

RADIOLOGIC STUDIES:  Radiologic studies also included a chest
CT, which confirmed cavitary lesions in the left lung apex
consistent with infectious process/tuberculosis.  This also
moderate-sized left pleural effusion.

HEAD CT:  Head CT showed no intracranial hemorrhage or mass
effect, but old infarction consistent with past medical
history.

ABDOMINAL CT:  Abdominal CT showed lesions of
T10 and sacrum most likely secondary to osteoporosis. These can
be followed by repeat imaging as an outpatient.



                            [**First Name8 (NamePattern2) **] [**First Name4 (NamePattern1) 1775**] [**Last Name (NamePattern1) **], M.D.  [**MD Number(1) 1776**]

Dictated By:[**Hospital 1807**]
MEDQUIST36

D:  [**2151-8-5**]  12:11
T:  [**2151-8-5**]  12:21
JOB#:  [**Job Number 1808**]


[...truncated]


--- Sample Note 2 ---

Admission Date:  [**2118-6-2**]       Dischar

In [ ]:
# Few-Shot Summarization of Clinical Notes
sample_note_1 = noteevents['TEXT'].dropna().iloc[0]
sample_note_2 = noteevents['TEXT'].dropna().iloc[1]
sample_note_3 = noteevents['TEXT'].dropna().iloc[2]
sample_note_4 = noteevents['TEXT'].dropna().iloc[3]

lay_summary_1 = (
    "Imaging tests showed signs of a lung infection, likely tuberculosis, with some fluid buildup around the lungs. "
    "Other scans found no brain bleeding and bone changes in the spine likely due to osteoporosis."
)

lay_summary_2 = (
    "An 81-year-old woman came to the hospital with breathing trouble, believed to be a COPD flare. "
    "She was on medications at home but didn't improve, so she went to the ER, where oxygen and IV steroids were given. "
    "She couldn't be taken off oxygen and needed further care in the ICU."
)

lay_summary_3 = (
    "An 81-year-old woman with severe COPD had trouble breathing for years. "
    "Doctors discovered that her airways were collapsing when she breathed out. "
    "She had surgery to support her windpipe and later had a breathing tube placed through her neck to help her breathe more easily."
)

demo_prompt = HUMAN_PROMPT + \
    "Here are examples of how to summarize clinical notes in plain language for a layperson:\n\n" + \
    "Clinical Note:\n" + sample_note_1 + "\n" + \
    "Layperson Summary:\n" + lay_summary_1 + "\n\n" + \
    "Clinical Note:\n" + sample_note_2 + "\n" + \
    "Layperson Summary:\n" + lay_summary_2 + "\n\n" + \
    "Clinical Note:\n" + sample_note_3 + "\n" + \
    "Layperson Summary:\n" + lay_summary_3 + "\n\n" + \
    "Now summarize this clinical note in plain language:\n\n" + \
    "Clinical Note:\n" + sample_note_4 + "\n" + \
    "Layperson Summary:\n"

resp = client.messages.create(
    model="claude-3-5-sonnet-20240620",
    max_tokens=200,
    messages=[{"role": "user", "content": demo_prompt}]
)

print("--- Layperson Summary ---")
print(resp.content[0].text)

--- Layperson Summary ---
Here is a summary of the clinical note in plain language:

An 87-year-old woman with a history of lung and heart problems was admitted to the hospital with severe shortness of breath. In the emergency room, she had trouble breathing and needed a breathing tube placed. She was treated in the intensive care unit with medications including steroids and antibiotics. 

While in the ICU, she developed bleeding in her stomach, which was examined with a camera procedure. This showed inflammation and ulcers in her esophagus and stomach. 

After about a week, doctors were able to remove the breathing tube and she slowly improved. She was moved to a regular hospital room where she continued to get better. By the time of discharge, she was able to breathe with just oxygen through nasal prongs.

The doctors made some changes to her medications, including adding stomach protection medicine, water pills, and tapering down steroids. They stopped some of her


In [34]:
# Chain of Thought

sample_note = noteevents['TEXT'].dropna().iloc[3] 

demo_prompt = HUMAN_PROMPT + (
    "Read the following clinical note carefully and write a detailed, step-by-step reasoning process based only on information in the note.\n"
    "Your reasoning should extract logical inferences, clarify the sequence of events, and explain connections between findings (labs, imaging, procedures, etc.).\n"
    "Avoid summarizing. Do not simplify into layperson language.\n"
    "Instead, aim to:\n"
    "1. Identify and explain abnormal findings (labs, imaging, physical exam).\n"
    "2. Trace clinical decisions and their justifications (e.g., why a procedure was ordered or repeated).\n"
    "3. Note progression or worsening of the condition over time.\n"
    "4. Include specifics like medication, ventilation, or surgical interventions as appropriate.\n\n"
    "Clinical Note:\n" + sample_note + "\n\n" +
    "Detailed Reasoning:\n"
)

resp = client.messages.create(
    model="claude-3-5-sonnet-20240620",
    max_tokens=700,
    messages=[{"role": "user", "content": demo_prompt}]
)

print("--- Detailed Step-by-Step Reasoning ---")
print(resp.content[0].text)

--- Detailed Step-by-Step Reasoning ---
Based on the clinical note, here is a detailed reasoning process:

1. Initial Presentation and Diagnosis:
   - 87-year-old female with history of COPD, CHF, and tracheobronchomalacia presented with acute dyspnea and lethargy.
   - Initial assessment in ED showed tachypnea, hypoxia (90% on 5L O2), and tripoding, consistent with COPD exacerbation.
   - Leukocytosis (WBC 14.1) suggested possible infection, but CXR did not show consolidation.
   - Initial management included nebulizers, steroids, and empiric antibiotics.

2. Progression to Respiratory Failure:
   - Patient's condition worsened with increased work of breathing and hypoxia despite non-invasive ventilation.
   - Intubation was required due to agitation, hypoxia (87%), and tachypnea (RR 40s).
   - Post-intubation ABG showed respiratory acidosis (pH 7.3, pCO2 60), confirming hypercarbic respiratory failure.

3. ICU Management:
   - Ventilator settings: 60% FiO2, PEEP 5, TV 450
   - Steroi

In [37]:
# In-Context Learning

sample_note_1 = noteevents['TEXT'].dropna().iloc[0]
sample_note_2 = noteevents['TEXT'].dropna().iloc[1]
sample_note_3 = noteevents['TEXT'].dropna().iloc[2]
sample_note_4 = noteevents['TEXT'].dropna().iloc[3]

demo_prompt = HUMAN_PROMPT + (
    "You are given clinical notes. Your task is to list detailed, step-by-step reasoning observations from each note, "
    "based only on the information in the text. Do not summarize or simplify.\n\n"

    "Clinical Note:\n" + sample_note_1 + "\n" +
    "Reasoning:\n"
    "1. Chest CT shows cavitary lesions in the left lung apex, suggesting infection (likely tuberculosis).\n"
    "2. Moderate left pleural effusion indicates fluid accumulation near infected lung.\n"
    "3. Head CT shows old infarcts, confirming known prior stroke history.\n"
    "4. Abdominal CT shows lesions of T10 and sacrum, likely due to osteoporosis.\n\n"

    "Clinical Note:\n" + sample_note_2 + "\n" +
    "Reasoning:\n"
    "1. Elderly patient with COPD presented with worsening shortness of breath for three days.\n"
    "2. Was initially treated at home with prednisone and levofloxacin, but symptoms persisted.\n"
    "3. Required home oxygen to maintain oxygen saturation >90%.\n"
    "4. At ER, placed on CPAP and treated with IV Solu-Medrol; still dependent on oxygen.\n"
    "5. Escalated to ICU care for respiratory support.\n\n"

    "Clinical Note:\n" + sample_note_3 + "\n" +
    "Reasoning:\n"
    "1. 81-year-old female with COPD had progressive dyspnea over 5 years.\n"
    "2. History of respiratory failure in previous admission required intubation.\n"
    "3. Bronchoscopy revealed severe tracheomalacia (airway collapse during exhalation).\n"
    "4. Underwent tracheobronchoplasty and tracheostomy after failed extubation.\n"
    "5. Tracheostomy later downsized to cuffless, suggesting stabilization.\n\n"

    "Clinical Note:\n" + sample_note_4 + "\n" +
    "Reasoning:\n"
)

resp = client.messages.create(
    model="claude-3-5-sonnet-20240620",
    max_tokens=700,
    messages=[{"role": "user", "content": demo_prompt}]
)

print("--- In-Context Learning Output (Reasoning for Sample Note 4) ---")
print(resp.content[0].text)


--- In-Context Learning Output (Reasoning for Sample Note 4) ---
Here are the key points from the clinical note:

1. 87-year-old female with history of COPD, CHF, and tracheobronchomalacia presented with acute dyspnea and lethargy.

2. In the ED, she became hypoxic and tachypneic, requiring intubation for hypercarbic respiratory failure. 

3. Workup revealed:
   - Leukocytosis
   - Parainfluenza virus type 3 on respiratory culture
   - No clear evidence of pneumonia on CXR
   - Echo showing pulmonary hypertension

4. Course complicated by:
   - Difficulty weaning from ventilator, required re-intubation after initial extubation attempt
   - Upper GI bleeding - EGD showed esophagitis, ulcers, hiatal hernia
   - MRSA colonization in sputum
   - Generalized weakness attributed to steroid myopathy and deconditioning

5. After discussions about goals of care, patient was successfully extubated without tracheostomy. 

6. Gradually improved and weaned to 2L oxygen (better than baseline 5L).

7

In [39]:
# Tree-of-Thought

sample_note_3 = noteevents['TEXT'].dropna().iloc[2]  

demo_prompt = (
    "You are a clinical reasoning assistant. Your task is to consider multiple reasoning paths to classify a patient's likely disposition.\n"
    "You will:\n"
    "1. Generate at least two plausible step-by-step reasoning paths from the clinical note.\n"
    "2. Evaluate which path is more consistent with the patient's course.\n"
    "3. Select the final disposition label: Home, ICU, Rehab, or Death.\n\n"

    "Clinical Note:\n" + sample_note_3 + "\n\n"

    "Thought Path 1:\n"
    "Step-by-step reasoning:\n"
    "[Start here...]"
)

resp1 = client.messages.create(
    model="claude-3-5-sonnet-20240620",
    max_tokens=350,
    messages=[{"role": "user", "content": demo_prompt}]
)
path_1_output = resp1.content[0].text

demo_prompt_path2 = (
    demo_prompt.replace("Thought Path 1", "Thought Path 2")
)

resp2 = client.messages.create(
    model="claude-3-5-sonnet-20240620",
    max_tokens=350,
    messages=[{"role": "user", "content": demo_prompt_path2}]
)
path_2_output = resp2.content[0].text

evaluation_prompt = (
    "You have two reasoning paths for the same clinical note.\n\n"
    "Thought Path 1:\n" + path_1_output + "\n\n" +
    "Thought Path 2:\n" + path_2_output + "\n\n" +
    "Evaluate both paths and decide which is more medically sound, complete, and consistent with the case.\n"
    "Then provide the final predicted disposition: Home, ICU, Rehab, or Death.\n\n"
    "Evaluation and Final Disposition:\n"
)

resp_eval = client.messages.create(
    model="claude-3-5-sonnet-20240620",
    max_tokens=300,
    messages=[{"role": "user", "content": evaluation_prompt}]
)

print("--- Tree of Thought Evaluation ---")
print(resp_eval.content[0].text)

--- Tree of Thought Evaluation ---
After evaluating both reasoning paths, I find that Thought Path 1 is more medically sound, complete, and consistent with the case. Here's why:

1. Thought Path 1 provides a more comprehensive overview of the patient's course, including specific dates for key events (tracheoplasty, tracheostomy).

2. It details the progression of the patient's condition, from initial complications to gradual improvement (ventilator weaning, transition to oral diet, ambulation progress).

3. It includes more specific medical details, such as the type of DVT (right upper extremity) and the anticoagulation treatment progression (heparin to Lovenox).

4. Thought Path 1 concludes with the actual discharge disposition, which Thought Path 2 does not explicitly state.

While Thought Path 2 does mention some additional comorbidities not found in Path 1, it is less complete in describing the patient's hospital course and outcome.

Based on the more comprehensive Thought Path 1, 

In [43]:
# Bonus - Classification Task

sample_note_1 = noteevents['TEXT'].dropna().iloc[0]
sample_note_2 = noteevents['TEXT'].dropna().iloc[1]
sample_note_3 = noteevents['TEXT'].dropna().iloc[2]
sample_note_4 = noteevents['TEXT'].dropna().iloc[3]  

demo_prompt = (
    "You are a medical assistant helping doctors anticipate patient disposition based on clinical notes.\n"
    "For each note, first reason through the clinical findings step-by-step. Then classify the most likely disposition:\n"
    "• Home\n• ICU\n• Rehab\n• Death\n\n"

    "Example 1:\n"
    "Clinical Note:\n" + sample_note_1 + "\n"
    "Reasoning:\n"
    "1. Imaging shows TB-like lung findings but no brain bleeding or acute trauma.\n"
    "2. Abnormal spine lesions likely due to osteoporosis; follow-up is outpatient.\n"
    "3. No mention of unstable vitals or life-threatening events.\n"
    "Disposition: Home\n\n"

    "Example 2:\n"
    "Clinical Note:\n" + sample_note_2 + "\n"
    "Reasoning:\n"
    "1. 81-year-old woman with COPD not responding to outpatient treatment.\n"
    "2. Required home oxygen, then ER CPAP and IV steroids.\n"
    "3. Failed weaning off CPAP and was transferred to the ICU.\n"
    "Disposition: ICU\n\n"

    "Example 3:\n"
    "Clinical Note:\n" + sample_note_3 + "\n"
    "Reasoning:\n"
    "1. Elderly patient with tracheomalacia and COPD requiring tracheoplasty.\n"
    "2. History of respiratory failure and multiple bronchoscopies.\n"
    "3. Tracheostomy later downsized, suggesting clinical stabilization.\n"
    "Disposition: Rehab\n\n"

    "Example 4:\n"
    "Clinical Note:\n" + sample_note_4 + "\n"
    "Reasoning:\n"
)

resp = client.messages.create(
    model="claude-3-5-sonnet-20240620",
    max_tokens=300,
    messages=[{"role": "user", "content": demo_prompt}]
)

print("--- Predicted Reasoning and Disposition ---")
print(resp.content[0].text)

--- Predicted Reasoning and Disposition ---
Based on the clinical note, here is my reasoning and disposition assessment:

1. 87-year-old female with history of COPD and tracheobronchomalacia 
2. Presented with acute dyspnea and respiratory failure requiring intubation
3. Had prolonged ICU stay with difficulty weaning from ventilator
4. Eventually extubated successfully after multiple attempts
5. Developed complications including GI bleed and generalized weakness
6. Required oxygen and nebulizer treatments at discharge
7. Discharged to extended care facility for rehabilitation
8. Multiple follow-up appointments scheduled with PCP and specialists

Given the patient's advanced age, prolonged hospitalization with complications, need for ongoing oxygen therapy, and discharge to a rehabilitation facility, the most appropriate disposition is:

Disposition: Rehab

The patient requires continued medical care and intensive rehabilitation before potentially returning home, making an extended care